# Interactive Tutorial

This notebook walks you through the core components of the thesis template.
Run each cell to explore the data, visualize samples, and understand the training pipeline.

**Prerequisites:** Make sure you have run `uv sync --all-extras` and are in the project root directory.

## 1. Setup

First, add the project root to the Python path and configure the environment.

In [1]:
import sys
from pathlib import Path

# Add project root to path (so we can import src.*)
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import os
os.environ.setdefault("DATA_DIR", "/mnt/data")

import torch
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend (works over SSH)
import matplotlib.pyplot as plt

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Project root: /home/USADR/ac142464/dev/mt_template
PyTorch version: 2.9.1+cu128
CUDA available: True


## 2. Configuration

The project uses [Hydra](https://hydra.cc/) for configuration. All configs live in `configs/`.
Let's inspect the main config and the dataset config.

In [2]:
import yaml

# Main config
with open(project_root / "configs" / "config.yaml") as f:
    main_config = yaml.safe_load(f)

print("=== Student fields (change these!) ===")
for key in ["student_name", "student_id", "work_type", "thesis_number"]:
    print(f"  {key}: {main_config[key]}")

print("\n=== Default selections ===")
for d in main_config["defaults"]:
    if isinstance(d, dict):
        for k, v in d.items():
            print(f"  {k}: {v}")

=== Student fields (change these!) ===
  student_name: CHANGE_ME
  student_id: CHANGE_ME
  work_type: CHANGE_ME
  thesis_number: CHANGE_ME

=== Default selections ===
  model: dummy
  dataset: ddacs
  training: fast_debug


In [3]:
# Dataset config
with open(project_root / "configs" / "dataset" / "ddacs.yaml") as f:
    dataset_config = yaml.safe_load(f)

print("=== Available modalities ===")
for name, info in dataset_config["modalities"].items():
    print(f"  {name:12s} -> {info['description']}")

print(f"\nDefault modality: {dataset_config['modality']}")
print(f"Total samples:    {dataset_config['metadata']['num_samples']}")

=== Available modalities ===
  h5           -> Raw H5 simulation files (uses DDACS package)
  pointcloud   -> Individual .pt files per sample with multi-timestep point clouds
  images       -> Image modality (not yet generated)
  graph        -> PyG Data objects with vertices, faces, edges
  mesh         -> PyG Data objects (same as graph modality)

Default modality: pointcloud
Total samples:    32071


## 3. Exploring the Data

The DDACS dataset contains deep drawing simulation results.
Each sample represents one simulation with different tool geometries and process parameters.

### 3.1 Load a single sample

In [4]:
from src.data.ddacs import PointCloudDDACSDataset

data_root = Path(os.environ["DATA_DIR"]) / "datasets" / "ddacs"
dataset = PointCloudDDACSDataset(root=str(data_root), split="train")

print(f"Training samples: {len(dataset)}")

# Load one sample
sample = dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Sample index: {sample['sample_index']}")
print(f"Geometry type: {sample['geometry']}")
print(f"Parameters: {sample['parameters']}")

Training samples: 25656


/home/USADR/ac142464/dev/mt_template/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Sample keys: ['data', 'parameters', 'geometry', 'sample_index']
Sample index: 16039
Geometry type: rectangular
Parameters: tensor([3.0000e+01, 9.0000e-01, 9.5000e-01, 5.0000e-02, 1.0000e+05])


### 3.2 Inspect the point cloud structure

Each sample contains a PyG Data object with multi-timestep point clouds for four tool components.

In [5]:
data = sample["data"]

# List all attributes in the Data object
print("=== Data attributes ===")
for key in data.keys():
    val = getattr(data, key)
    if isinstance(val, torch.Tensor):
        print(f"  {key:20s} shape={str(list(val.shape)):20s} dtype={val.dtype}")
    else:
        print(f"  {key:20s} {type(val).__name__}")

=== Data attributes ===
  die                  shape=[3, 2048, 3]         dtype=torch.float32
  binder               shape=[3, 512, 3]          dtype=torch.float32
  norm_scale           shape=[1]                  dtype=torch.float32
  blank                shape=[4, 4096, 3]         dtype=torch.float32
  blank_thickness      shape=[4, 4096]            dtype=torch.float32
  blank_strain         shape=[4, 4096, 2, 6]      dtype=torch.float32
  norm_offset          shape=[3]                  dtype=torch.float32
  blank_stress         shape=[4, 4096, 3, 6]      dtype=torch.float32
  punch                shape=[3, 2048, 3]         dtype=torch.float32


### 3.3 Parameter distributions

The 5 process parameters control the deep drawing simulation.

In [6]:
import pandas as pd

metadata = pd.read_csv(data_root / "metadata2.csv")

param_cols = [
    "curvature_radius",
    "material_scaling_factor",
    "sheet_metal_thickness",
    "friction_coefficient",
    "blankholder_force",
]

print(metadata[param_cols].describe().round(4).to_string())

       curvature_radius  material_scaling_factor  sheet_metal_thickness  friction_coefficient  blankholder_force
count        32071.0000               32071.0000             32071.0000            32071.0000         32071.0000
mean            86.6739                   1.0000                 0.9750                0.1000        300020.2675
std             49.8900                   0.0645                 0.0171                0.0316        129097.9337
min             30.0000                   0.9000                 0.9500                0.0500        100000.0000
25%             40.0000                   0.9500                 0.9600                0.0700        200000.0000
50%            100.0000                   1.0000                 0.9800                0.1000        300000.0000
75%            150.0000                   1.0500                 0.9900                0.1300        400000.0000
max            150.0000                   1.1000                 1.0000                0.1500   

## 4. Visualization

The template provides ready-to-use visualization functions for both point clouds and meshes.

### 4.1 Point cloud visualization

In [7]:
from src.visualization import plot_pointclouds, setup_plot_for_thesis, SIZES

# Load legacy point cloud format for visualization
pc_dir = data_root / "pointcloud"
vis_sample = {
    "blank": torch.load(pc_dir / "blanks.pt", weights_only=False)[0],
    "die": torch.load(pc_dir / "dies.pt", weights_only=False)[0],
    "punch": torch.load(pc_dir / "punches.pt", weights_only=False)[0],
    "binder": torch.load(pc_dir / "binders.pt", weights_only=False)[0],
}

fig = plot_pointclouds(vis_sample, title="Point Cloud Sample", show=False)
plt.show()

### 4.2 Mesh visualization

In [8]:
from src.visualization import plot_meshes

# Load graph/mesh data
graph_dir = data_root / "graph"
mesh_sample = {
    "blank": torch.load(graph_dir / "blanks.pt", weights_only=False)[0],
    "die": torch.load(graph_dir / "dies.pt", weights_only=False)[0],
    "punch": torch.load(graph_dir / "punches.pt", weights_only=False)[0],
    "binder": torch.load(graph_dir / "binders.pt", weights_only=False)[0],
}

fig = plot_meshes(mesh_sample, title="Mesh Sample", show=False)
plt.show()

### 4.3 Figure sizes

Use the pre-configured sizes for consistent figures in your thesis or paper.

In [9]:
from src.visualization import SIZES

print("=== Thesis (DIN A4) ===")
for name, size in SIZES["thesis"].items():
    print(f"  {name:10s} -> {size[0]:.1f} x {size[1]:.1f} inches")

print("\n=== Paper (IEEE two-column) ===")
for name, size in SIZES["paper"].items():
    print(f"  {name:10s} -> {size[0]:.2f} x {size[1]:.1f} inches")

=== Thesis (DIN A4) ===
  full       -> 6.3 x 2.1 inches
  half       -> 3.0 x 2.1 inches
  grid_2x2   -> 6.3 x 5.0 inches

=== Paper (IEEE two-column) ===
  single_col -> 3.50 x 2.5 inches
  double_col -> 7.16 x 2.5 inches
  grid_2x2   -> 7.16 x 5.0 inches


## 5. Model Architecture

All models inherit from `BaseSurrogateModel`. The `DummyModel` shows the pattern you should follow.

In [10]:
from src.models.dummy_model import DummyModel

# Create model from config dict
model_config = {
    "architecture": {
        "input_dim": 10,
        "hidden_dims": [128, 64, 32],
        "output_dim": 3,
        "activation": "relu",
        "dropout": 0.1,
        "batch_norm": True,
    },
    "optimizer": {
        "name": "adam",
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
    },
}

model = DummyModel(model_config)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

DummyModel(
  (model): Sequential(
    (0): Linear(in_features=10, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.1, inplace=False)
    (12): Linear(in_features=32, out_features=3, bias=True)
  )
)

Total parameters: 12,291


### Creating your own model

Copy the dummy model and modify:

```bash
cp src/models/dummy_model.py src/models/my_model.py
cp configs/model/dummy.yaml configs/model/my_model.yaml
```

Your model must implement:
- `forward(x)` - the forward pass
- `configure_optimizers()` - optimizer (and optionally scheduler)
- `preprocess_data(batch)` - convert raw batch to `(inputs, targets)`

See `src/models/base_model.py` for all available hooks.

## 6. Training

Training is launched from the command line using Hydra:

```bash
# Quick test (3 epochs)
uv run main.py model=dummy dataset=ddacs training=fast_debug

# Override parameters
uv run main.py model=dummy dataset=ddacs training.max_epochs=10 model.optimizer.learning_rate=0.0005

# Switch data modality
uv run main.py model=dummy dataset=ddacs +dataset.modality=graph
```

All metrics and artifacts are automatically logged to MLflow.

## 7. MLflow: Viewing Results

After training, inspect your results in the MLflow UI:

```bash
uv run mlflow ui --backend-store-uri sqlite:///./mlruns/mlflow.db --port 5000
```

Or query results programmatically:

In [ ]:
import mlflow

# Point MLflow to local tracking
tracking_uri = f"sqlite:///{project_root}/mlruns/mlflow.db"
mlflow.set_tracking_uri(tracking_uri)

client = mlflow.tracking.MlflowClient()

# List experiments
experiments = client.search_experiments()
for exp in experiments:
    print(f"Experiment: {exp.name} (ID: {exp.experiment_id})")

# List recent runs (if any exist)
runs = client.search_runs(experiment_ids=[exp.experiment_id for exp in experiments])
if runs:
    for run in runs[:3]:
        print(f"\nRun: {run.info.run_id[:8]}... Status: {run.info.status}")
        for k, v in run.data.metrics.items():
            print(f"  {k}: {v:.4f}")
else:
    print("\nNo runs yet. Run a training first!")

## 8. Evaluation and Submission

After training, evaluate and register your best model:

```bash
# Evaluate on all datasets
uv run evaluate.py --run_id <run_id> --datasets all --register

# Validate your submission
uv run pytest tests/test_submission.py -v -m "not slow"
```

The test suite checks that your model:
- Inherits from `BaseSurrogateModel`
- Has a valid config file
- Produces correct output shapes
- Is registered in MLflow

## 9. Next Steps

1. **Configure** your student info in `configs/config.yaml`
2. **Explore** the data using this notebook
3. **Copy** the dummy model: `cp src/models/dummy_model.py src/models/my_model.py`
4. **Implement** your architecture in `forward()` and `preprocess_data()`
5. **Train** with `uv run main.py model=my_model dataset=ddacs`
6. **Iterate** on your model, checking MLflow for results
7. **Evaluate** with `uv run evaluate.py --run_id <id> --datasets all --register`
8. **Validate** with `uv run pytest tests/test_submission.py -v`

For detailed documentation, see the other pages in the Sphinx docs or run `make livehtml` in the `docs/` directory.